### 

In [ ]:
import gzip
import json
from collections import defaultdict
from tqdm import tqdm

def load_user2index(file_path):
    user_dict = {}
    with open(file_path, 'r') as fin:
        lines = fin.readlines()
        for line in lines[1:]:
            user_index, user_id = line.strip().split()
            user_dict[user_id] = int(user_index) 
    return user_dict

def parse_reviews(path, user_dict):
    g = gzip.open(path, 'rb')
    User = defaultdict(list)
    rating_non = 0
    for l in tqdm(g, desc="Processing reviews"):
        review_data = json.loads(l)
        asin = review_data['asin']
        rev = review_data['reviewerID']
        time = review_data['unixReviewTime']
        reviewText = review_data.get('reviewText',' ')
        summary = review_data.get('summary',' ')
        rating = review_data.get('overall','nan')
        if str(rating) == "nan":
            rating_non += 1
            rating = 0
        

        user_id = user_dict.get(rev) 
        
        if user_id is not None: 
            User[user_id].append({
                'time': time,
                'itemid': asin,
                'reviewText': reviewText,
                'summary': summary,
                'rating': rating,
                'reviewID': rev
            })
    print(rating_non)
    return User


file_path_reviews = '/root/autodl-tmp/music/CDs_and_Vinyl_5.json.gz'
file_path_user2index = '/root/autodl-tmp/music/user2index.txt'

user_dict = load_user2index(file_path_user2index)  
User = parse_reviews(file_path_reviews, user_dict)  


print("Review processing complete.")


Processing reviews: 1443755it [00:40, 35884.28it/s]

0
Review processing complete.


In [16]:
User[0]

[{'time': 1488931200,
  'itemid': 'B000002B7Y',
  'reviewText': 'Great CD',
  'summary': ' ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1487030400,
  'itemid': 'B000002KKK',
  'reviewText': 'Good CD from that threesome from Texas. ',
  'summary': 'Zztop ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1487030400,
  'itemid': 'B000002KJS',
  'reviewText': 'Zztop is awesome ',
  'summary': 'Great CD!!!',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1487376000,
  'itemid': 'B000006RHL',
  'reviewText': 'Classic ',
  'summary': 'Classic concert CD!!!!',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1488931200,
  'itemid': 'B0000251VP',
  'reviewText': 'Awesome',
  'summary': ' ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1445212800,
  'itemid': 'B003OF3R0S',
  'reviewText': 'Awesome cd',
  'summary': 'Awesome cd!!!!!',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1487030400,
  'itemid': 'B0041ON36

### 

In [ ]:
import gzip
import json
import pandas as pd
from tqdm import tqdm


def load_item2index(file_path):
    item_dict = {}
    with open(file_path, 'r') as fin:
        lines = fin.readlines()
        for line in lines[1:]:
            item_index, item_id = line.strip().split()
            item_dict[item_id] = int(item_index) 
    return item_dict


def parse_meta(path, item_dict):
    Item = defaultdict(list)
    
    with gzip.open(path, 'rt', encoding='utf-8') as fin:
        for line in tqdm(fin, desc="Processing meta data"):
            meta_data = json.loads(line)
            asin = meta_data['asin']
            title = meta_data.get('title', ' ')
            description = meta_data.get('description', ' ')
            
            
            item_id = item_dict.get(asin)  
            
            if item_id is not None: 
                Item[item_id].append({
                    'title': title,
                    'description': description,
                    'item_id': asin
                })
    
    return Item


file_path_meta = '/root/autodl-tmp/music/meta_CDs_and_Vinyl.json.gz'
file_path_item2index = '/root/autodl-tmp/music/item2index.txt'

item_dict = load_item2index(file_path_item2index)  
Item = parse_meta(file_path_meta, item_dict) 


print("Meta processing complete.")


Processing meta data: 516914it [00:16, 30928.61it/s]

Meta processing complete.


In [18]:
Item[0]

[{'title': 'Christmas Eve and Other Stories',
  'description': ['This is a concept album all the way, with tales of Christmas kindness intertwined with masterful musicianship as they play traditionals plus their own An Angel Came Down; First Snow; Ornament; Old City Bar , and more!',
   'Is the Trans-Siberian Orchestra\'s <I>Christmas Eve and Other Stories</I> a holiday rock opera? Or perhaps just a holiday prog-rock disc? Or maybe it\'s New Age? Whatever the case may be, this isn\'t your typical Christmas album. Filled with electric guitar solos, plenty of synthesized keyboards, a children\'s choir, and lively drumming, <I>Christmas Eve</I> can only be compared to one other record, the Trans-Siberian Orchestra\'s <I>other</I> holiday disc, <I><a href="/exec/obidos/ASIN/B00000AEDW/${0}">The Christmas Attic</a></I>. On this CD, angelic vocal solos (on numbers such as "The Prince of Peace") are interspersed with driving instrumentals. Sentimental, occasionally bombastic, but as high-conc

### 

In [ ]:

def load_train_data(file_path):
    train_data = defaultdict(set)
    with open(file_path, 'r') as fin:
        lines = fin.readlines()
        for line in lines[1:]: 
            user_index, item_index, _ = line.strip().split()
            train_data[int(user_index)].add(int(item_index))  # user_index -> set of item_indexes
    return train_data

def filter_user_data(User, train_data, item_dict):
    for user_id in list(User.keys()):  
        valid_item_ids = train_data.get(user_id, set())  
        User[user_id] = [review for review in User[user_id] if item_dict.get(review['itemid']) in valid_item_ids]
        
        
        if not User[user_id]:
            del User[user_id]
    return User


In [20]:
file_path_train = "/root/autodl-tmp/music/train.txt"
train_data = load_train_data(file_path_train)
User = filter_user_data(User, train_data, item_dict)

In [21]:
User[0]

[{'time': 1488931200,
  'itemid': 'B000002B7Y',
  'reviewText': 'Great CD',
  'summary': ' ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1487030400,
  'itemid': 'B000002KKK',
  'reviewText': 'Good CD from that threesome from Texas. ',
  'summary': 'Zztop ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1487376000,
  'itemid': 'B000006RHL',
  'reviewText': 'Classic ',
  'summary': 'Classic concert CD!!!!',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1488931200,
  'itemid': 'B0000251VP',
  'reviewText': 'Awesome',
  'summary': ' ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1445212800,
  'itemid': 'B00DLKXF3A',
  'reviewText': 'Killer CD!!!!!!!',
  'summary': 'Killer awesome cd !!!',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1445212800,
  'itemid': 'B00KWN13BU',
  'reviewText': 'Awesome CD!!!!!!!',
  'summary': 'Awesome cd!!',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV'},
 {'time': 1445212800,
  'itemid

In [22]:
len(User)

15404

###

In [ ]:
def add_title_to_reviews(User, Item):
    empty_title_count = 0
    for user_id, reviews in User.items():
        for review in reviews:
            asin = review['itemid']
            
            item_index = item_dict.get(asin)
            
            item_info = Item.get(asin)
            if item_index is not None:
                item_info = Item.get(item_index)
                if item_info:
                    title = item_info[0].get('title', ' ').strip()
                    description = item_info[0].get('description', ' ')
                    if isinstance(description, list):  
                        description = " ".join(description).strip()
                    else: 
                        description = description.strip()
                        
                    if any(len(word) > 50 for word in title.split()):
                        title = description
                        
                    
                    if title:
                        review['title'] = title
                    elif description:
                        if any(len(word) > 50 for word in description.split()):
                            description = 'none'
                            empty_title_count += 1
                        review['title'] = description
                    else:
                        review['title'] = 'none'
                        empty_title_count += 1
                else:
                    review['title'] = 'none'
                    empty_title_count += 1
    
    return empty_title_count

In [ ]:

empty_title_count = add_title_to_reviews(User, Item)

In [25]:
User[0]

[{'time': 1488931200,
  'itemid': 'B000002B7Y',
  'reviewText': 'Great CD',
  'summary': ' ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV',
  'title': 'No Rest for the Wicked'},
 {'time': 1487030400,
  'itemid': 'B000002KKK',
  'reviewText': 'Good CD from that threesome from Texas. ',
  'summary': 'Zztop ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV',
  'title': 'Deguello'},
 {'time': 1487376000,
  'itemid': 'B000006RHL',
  'reviewText': 'Classic ',
  'summary': 'Classic concert CD!!!!',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV',
  'title': 'Live After Death'},
 {'time': 1488931200,
  'itemid': 'B0000251VP',
  'reviewText': 'Awesome',
  'summary': ' ',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV',
  'title': 'Iron Maiden enhanced  eng'},
 {'time': 1445212800,
  'itemid': 'B00DLKXF3A',
  'reviewText': 'Killer CD!!!!!!!',
  'summary': 'Killer awesome cd !!!',
  'rating': 5.0,
  'reviewID': 'A100BBKBDGGRHV',
  'title': 'Hail To The King Deluxe'},
 {'time': 1445212800,
  'i

In [26]:
empty_title_count

5

### 得到prompt

In [ ]:
def create_interaction_prompt(User, filename):
    prompts = []
    for userid in User:
        # Sort by unixReviewTime from high to low
        User[userid] = sorted(User[userid], key=lambda x: x['time'], reverse=True)

        total_words = 0
        reviews = []
        for entry in User[userid]:
            review_text = entry["reviewText"]
            words = review_text.split()
            if len(words) > 200:
                truncated_review = ' '.join(words[:200])
                summary = entry.get("summary", "")
                if summary:
                    truncated_review += ". The summary:" + summary
            else:
                truncated_review = review_text
                summary = entry.get("summary", "")
                if summary:
                    truncated_review += ". The summary:" + summary

            review_length = len(truncated_review.split())
            title = entry.get("title", "")

            # Handle the case where title is a list
            if isinstance(title, list):
                title = " ".join(title)

            if title:
                title_len = len(title.split())
            else:
                title_len = 0

            if title_len + total_words + review_length > 2500:
                break

            reviews.append(f'{{"title": "{title}", "review": "{truncated_review}"}}')
            total_words = total_words + review_length + title_len

        prompt = f'userid: {userid}\nInteracted items: \n[\n' + ',\n'.join(reviews) + '\n]'
        prompts.append({"userid": int(userid),"prompt": prompt})
        
    prompts = sorted(prompts, key=lambda x: x["userid"])
    # Save the prompts to the specified filename
    with open(filename, 'w') as f:
        json.dump(prompts, f, indent=4)

In [28]:
create_interaction_prompt(User, "/root/autodl-tmp/music/crossaug_music_train.json")